# SO Fisher constraint analysis

This notebook analyzes the completed Fisher products in this directory for SO baseline and goal noise, each with conditional-noise and Gaussian-total covariance.

**Important:** every data Fisher matrix has rank 6/9, so a data-only nine-parameter marginalized covariance is singular. The early sensitivity panels use the saved Gaussian moment-prior regularization and are labeled accordingly. The final Battaglia12 comparison instead applies the exact hard bounded uniform SBI prior by importance sampling.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.lines import Line2D

plt.rcParams.update({
    "font.family": "serif", "mathtext.fontset": "cm", "font.size": 8,
    "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7,
    "ytick.labelsize": 7, "legend.fontsize": 7, "pdf.fonttype": 42,
    "ps.fonttype": 42, "savefig.bbox": "tight",
})

relative = Path("SBI_analysis/adrian_fisher_baseline_deproj0/2_param_fisher_analysis")
candidates = [Path.cwd(), Path.cwd()/relative, Path.cwd().parent/"2_param_fisher_analysis"]
ROOT = next((p.resolve() for p in candidates if (p/"fisher_sensitivity_complete.json").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the notebook directory or HalfDome repository root.")
FIG = ROOT/"notebook_figures"
FIG.mkdir(exist_ok=True)

CASES = ("baseline", "goal")
SCOPES = ("conditional_noise", "gaussian_total")
CASE_LABEL = {"baseline": "SO baseline", "goal": "SO goal"}
SCOPE_LABEL = {"conditional_noise": "Conditional noise", "gaussian_total": "Gaussian total"}
CASE_COLOR = {"baseline": "#d62728", "goal": "#1f77b4"}
SCOPE_COLOR = {"conditional_noise": "#2ca02c", "gaussian_total": "#9467bd"}
PARAM_FALLBACK = ["P0","xc","beta","alpha_m_P0","alpha_m_xc","alpha_m_beta",
                  "alpha_z_P0","alpha_z_xc","alpha_z_beta"]
FID_FALLBACK = np.array([18.1,0.497,4.35,0.154,-0.00865,0.0393,-0.758,0.731,0.415])
LABEL = {
 "P0":r"$P_0$","xc":r"$x_{\rm c}$","beta":r"$\beta$",
 "alpha_m_P0":r"$\alpha_{m,P_0}$","alpha_m_xc":r"$\alpha_{m,x_{\rm c}}$",
 "alpha_m_beta":r"$\alpha_{m,\beta}$","alpha_z_P0":r"$\alpha_{z,P_0}$",
 "alpha_z_xc":r"$\alpha_{z,x_{\rm c}}$","alpha_z_beta":r"$\alpha_{z,\beta}$"}
print("Output root:", ROOT)
print("Figure directory:", FIG)

In [ ]:
def find_col(frame, choices):
    lookup = {str(c).lower(): c for c in frame.columns}
    return next((lookup[x.lower()] for x in choices if x.lower() in lookup), None)

def parameter_values(frame, names, choices):
    value_col = find_col(frame, choices)
    if value_col is None:
        return None
    name_col = find_col(frame, ["parameter","param","parameter_name","param_name","name"])
    index_col = find_col(frame, ["parameter_index","param_index","index"])
    work = frame.sort_values(index_col) if index_col else frame
    if name_col:
        mapping = {str(r[name_col]): float(r[value_col]) for _,r in work.iterrows()
                   if pd.notna(r[value_col])}
        if all(n in mapping for n in names):
            return np.array([mapping[n] for n in names])
    values = pd.to_numeric(work[value_col], errors="coerce").dropna().to_numpy()
    return values[:len(names)] if len(values) >= len(names) else None

results = {}
for case in CASES:
    for scope in SCOPES:
        directory = ROOT/case/scope
        fisher = np.load(directory/"fisher_normalized_by_prior_width.npy")
        cov_q = np.load(directory/"fisher_prior_regularized_covariance_normalized.npy")
        eigenvalues = np.load(directory/"fisher_eigenvalues.npy")
        identified_file = directory/"fisher_identified_modes.npy"
        identified = (np.load(identified_file).astype(bool) if identified_file.exists()
                      else eigenvalues > np.max(np.abs(eigenvalues))*1e-10)
        results[(case,scope)] = {
            "sens": pd.read_csv(directory/"parameter_sensitivities.csv"),
            "cov_b": np.load(directory/"covariance_binned_dell.npy"),
            "corr_b": np.load(directory/"correlation_binned_dell.npy"),
            "fisher_q": (fisher+fisher.T)/2,
            "cov_q": (cov_q+cov_q.T)/2,
            "eigenvalues": eigenvalues,
            "identified": identified,
            "estimable": np.load(directory/"fisher_estimable_fraction.npy"),
        }

meta = results[("baseline","conditional_noise")]["sens"].copy()
name_col = find_col(meta, ["parameter","param","parameter_name","param_name","name"])
index_col = find_col(meta, ["parameter_index","param_index","index"])
meta = meta.sort_values(index_col) if index_col else meta
names = [str(v) for v in meta[name_col].drop_duplicates().tolist()[:9]] if name_col else PARAM_FALLBACK
names = names if len(names)==9 else PARAM_FALLBACK
fid = parameter_values(meta,names,["fiducial","fiducial_value","theta_fiducial","truth","theta_true"])
prior_lo = parameter_values(meta,names,["prior_low","prior_min","lower","theta_min"])
prior_hi = parameter_values(meta,names,["prior_high","prior_max","upper","theta_max"])
prior_width = parameter_values(meta,names,["prior_width","delta_prior","prior_range"])
fid = FID_FALLBACK if fid is None else fid
if prior_width is None and prior_lo is not None and prior_hi is not None:
    prior_width = prior_hi-prior_lo
if prior_width is None:
    raise KeyError("Prior widths are absent from parameter_sensitivities.csv")
prior_lo = fid-prior_width/2 if prior_lo is None else prior_lo
prior_hi = fid+prior_width/2 if prior_hi is None else prior_hi
idx = {name:i for i,name in enumerate(names)}
labels = [LABEL.get(name,name) for name in names]
scale = np.diag(prior_width)
for result in results.values():
    result["cov_theta"] = scale@result["cov_q"]@scale

rows = []
for (case,scope),result in results.items():
    keep = result["identified"]
    positive = result["eigenvalues"][keep]
    rows.append({"noise":CASE_LABEL[case],"covariance":SCOPE_LABEL[scope],
                 "rank":int(keep.sum()),"parameters":len(names),
                 "identified_condition":positive.max()/positive.min()})
rank_table = pd.DataFrame(rows)
print(rank_table.to_string(index=False))
print("\nParameters:", names)
print("Fiducial:", fid)
print("Prior widths:", prior_width)

## Covariance construction

The statistic is the masked, binned pseudo-\(D_\ell\) split cross-spectrum. The pipeline uses the SO deprojection-0 \(N_\ell\), \(f_{\rm sky}=0.4\), the SBI multipole range, and the same \(2\ell+1\) bin weighting.

For two independent noise splits, conditional-noise covariance holds the Battaglia12 signal fixed:

\[
{\rm Var}(\widetilde C_\ell^{AB}\mid s)=
\frac{2\widetilde C_\ell\widetilde N_\ell+\widetilde N_\ell^2}
{(2\ell+1)f_{\rm sky}}.
\]

Gaussian-total covariance also includes Gaussian signal sample variance:

\[
{\rm Var}(\widetilde C_\ell^{AB})=
\frac{(\widetilde C_\ell+\widetilde N_\ell)^2+\widetilde C_\ell^2}
{(2\ell+1)f_{\rm sky}}.
\]

The binning matrix includes \(D_\ell=\ell(\ell+1)C_\ell/(2\pi)\) and \(2\ell+1\) weighting:
\[
C_{bb'}=\sum_\ell B_{b\ell}{\rm Var}(\widetilde C_\ell)B_{b'\ell}.
\]
For \(J_{ib}=\partial D_b/\partial\theta_i\), \(F=JC^{-1}J^T\).

The saved matrix uses \(q_i=(\theta_i-\theta_{i,\rm fid})/\Delta\theta_{i,\rm prior}\). Since the rank is 6/9, the early sensitivity plots use
\[
{\rm Cov}(q)=(F_q+12I)^{-1},
\]
where \(12I\) is the precision of a Gaussian with the same variance, \(1/12\), as a unit-width uniform prior. Null-direction widths in those early plots are therefore prior-driven. The final Battaglia12 comparison does not interpret this Gaussian as the prior: it evaluates the observation-shifted linear likelihood and importance-samples the exact hard bounded uniform SBI prior.

The forecast omits exact mask mode coupling, connected tSZ trispectrum, and super-sample covariance. Conditional-noise is narrower by construction because it excludes signal sample variance.

In [ ]:
cov_rows = []
for (case,scope),result in results.items():
    eig = np.linalg.eigvalsh(result["cov_b"])
    positive = eig[eig>0]
    correlation = result["corr_b"]
    cov_rows.append({
        "noise":CASE_LABEL[case],"covariance":SCOPE_LABEL[scope],
        "bins":len(correlation),"min_variance":np.diag(result["cov_b"]).min(),
        "max_variance":np.diag(result["cov_b"]).max(),
        "condition":positive.max()/positive.min(),
        "max_abs_offdiag_correlation":np.max(np.abs(correlation-np.eye(len(correlation))))})
covariance_table = pd.DataFrame(cov_rows)
print(covariance_table.to_string(index=False))

fig,axes = plt.subplots(1,2,figsize=(18/2.54,6.5/2.54))
for ax,scope in zip(axes,SCOPES):
    for case in CASES:
        sigma = np.sqrt(np.diag(results[(case,scope)]["cov_b"]))
        ax.plot(np.arange(len(sigma)),sigma,"o-",ms=2.5,lw=1,
                color=CASE_COLOR[case],label=CASE_LABEL[case])
    ax.set_yscale("log")
    ax.set_xlabel("Multipole-bin index")
    ax.set_title(SCOPE_LABEL[scope])
    ax.grid(alpha=0.25)
axes[0].set_ylabel(r"$\sqrt{{\rm Cov}(D_b,D_b)}$")
axes[0].legend(frameon=False)
fig.tight_layout()
covariance_path = FIG/"binned_covariance_scale.jpg"
fig.savefig(covariance_path,dpi=300)
plt.show()
print("Saved:",covariance_path)

In [ ]:
LEVEL = {0.68:2.30,0.95:5.99}

def pair_cov(result,i,j):
    covariance = result["cov_theta"][np.ix_([i,j],[i,j])]
    covariance = (covariance+covariance.T)/2
    minimum = np.linalg.eigvalsh(covariance).min()
    if minimum<=0:
        covariance += np.eye(2)*(abs(minimum)+1e-14)
    return covariance

def add_ellipse(ax,mean,covariance,probability,color,filled=False,alpha=1,ls="-",lw=1.2):
    values,vectors = np.linalg.eigh(covariance)
    order = np.argsort(values)[::-1]
    values,vectors = np.maximum(values[order],0),vectors[:,order]
    angle = np.degrees(np.arctan2(vectors[1,0],vectors[0,0]))
    width,height = 2*np.sqrt(LEVEL[probability]*values)
    ax.add_patch(Ellipse(mean,width,height,angle=angle,edgecolor=color,
                         facecolor=color if filled else "none",
                         alpha=alpha,ls=ls,lw=lw))

def limits(result,i,nsigma=3.8):
    sigma = np.sqrt(result["cov_theta"][i,i])
    low = max(prior_lo[i],fid[i]-nsigma*sigma)
    high = min(prior_hi[i],fid[i]+nsigma*sigma)
    if high<=low:
        low,high = prior_lo[i],prior_hi[i]
    pad = 0.04*(high-low)
    return low-pad,high+pad

groups = {
    "profile_shape":["P0","xc","beta"],
    "mass_evolution":["alpha_m_P0","alpha_m_xc","alpha_m_beta"],
    "redshift_evolution":["alpha_z_P0","alpha_z_xc","alpha_z_beta"]}

def group_triangle(case,scope,group):
    result,selected = results[(case,scope)],groups[group]
    ids,color = [idx[name] for name in selected],CASE_COLOR[case]
    fig,axes = plt.subplots(3,3,figsize=(18/2.54,18/2.54))
    for row in range(3):
        for column in range(3):
            ax,i,j = axes[row,column],ids[column],ids[row]
            if row<column:
                ax.set_visible(False)
                continue
            if row==column:
                sigma = np.sqrt(result["cov_theta"][i,i])
                x = np.linspace(*limits(result,i),500)
                y = np.exp(-0.5*((x-fid[i])/sigma)**2)
                ax.plot(x,y,color=color,lw=1.4)
                ax.fill_between(x,0,y,color=color,alpha=0.22)
                ax.axvline(fid[i],color="black",ls=":",lw=0.9)
                ax.set_xlim(*limits(result,i))
                ax.set_ylim(0,1.08)
                ax.set_yticks([])
            else:
                covariance = pair_cov(result,i,j)
                mean = [fid[i],fid[j]]
                add_ellipse(ax,mean,covariance,0.95,color,True,0.14,lw=0.9)
                add_ellipse(ax,mean,covariance,0.68,color,True,0.32,lw=1.2)
                ax.axvline(fid[i],color="black",ls=":",lw=0.8)
                ax.axhline(fid[j],color="black",ls=":",lw=0.8)
                ax.set_xlim(*limits(result,i))
                ax.set_ylim(*limits(result,j))
            if row==2:
                ax.set_xlabel(labels[i])
            else:
                ax.set_xticklabels([])
            if column==0 and row>0:
                ax.set_ylabel(labels[j])
            elif row>column:
                ax.set_yticklabels([])
            ax.grid(alpha=0.16)
    fig.suptitle(f"{CASE_LABEL[case]}, {SCOPE_LABEL[scope]}: {group.replace('_',' ')}\n"
                 "9D prior-regularized Fisher covariance",y=0.995)
    fig.tight_layout(rect=(0,0,1,0.95))
    output = FIG/f"triangle_{case}_{scope}_{group}.jpg"
    fig.savefig(output,dpi=300)
    plt.close(fig)
    return output

triangle_paths = [group_triangle(case,scope,group)
                  for case in CASES for scope in SCOPES for group in groups]
print("Saved",len(triangle_paths),"separate triangle plots:")
print("\n".join(str(path) for path in triangle_paths))

## Main \(P_0\)-\(\beta\) contours

Each noise implementation is kept in a separate figure. Its two panels compare covariance assumptions without overplotting. These are submatrices of the full nine-parameter prior-regularized covariance, so the other seven parameters are marginalized, not fixed.

In [ ]:
def p0_beta(case):
    i,j = idx["P0"],idx["beta"]
    fig,axes = plt.subplots(1,2,figsize=(18/2.54,7.5/2.54))
    for ax,scope in zip(axes,SCOPES):
        result,color = results[(case,scope)],SCOPE_COLOR[scope]
        covariance,mean = pair_cov(result,i,j),[fid[i],fid[j]]
        add_ellipse(ax,mean,covariance,0.95,color,True,0.16,lw=1.0)
        add_ellipse(ax,mean,covariance,0.68,color,True,0.36,lw=1.4)
        ax.axvline(fid[i],color="black",ls=":",lw=0.9)
        ax.axhline(fid[j],color="black",ls=":",lw=0.9)
        ax.plot(fid[i],fid[j],"k+",ms=6)
        ax.set_xlim(*limits(result,i))
        ax.set_ylim(*limits(result,j))
        ax.set_xlabel(labels[i])
        ax.set_title(SCOPE_LABEL[scope])
        ax.grid(alpha=0.2)
    axes[0].set_ylabel(labels[j])
    fig.suptitle(f"{CASE_LABEL[case]}: marginalized $P_0$-$\\beta$ constraints\n"
                 "9D prior-regularized Fisher covariance",y=1.02)
    fig.tight_layout()
    output = FIG/f"p0_beta_constraints_{case}.jpg"
    fig.savefig(output,dpi=300)
    plt.show()
    return output

main_paths = [p0_beta(case) for case in CASES]
print("Saved:",*main_paths,sep="\n")

In [ ]:
pairs = [("P0","beta"),("P0","xc"),("xc","beta"),
         ("alpha_m_P0","alpha_z_P0")]

def compare_noise(scope):
    fig,axes = plt.subplots(2,2,figsize=(18/2.54,16/2.54))
    for ax,(xname,yname) in zip(axes.flat,pairs):
        i,j = idx[xname],idx[yname]
        all_x,all_y = [],[]
        for case in CASES:
            result,color = results[(case,scope)],CASE_COLOR[case]
            covariance,mean = pair_cov(result,i,j),[fid[i],fid[j]]
            add_ellipse(ax,mean,covariance,0.95,color,False,0.85,"--",0.9)
            add_ellipse(ax,mean,covariance,0.68,color,False,1.0,"-",1.4)
            all_x += [fid[i]-3.8*np.sqrt(covariance[0,0]),
                      fid[i]+3.8*np.sqrt(covariance[0,0])]
            all_y += [fid[j]-3.8*np.sqrt(covariance[1,1]),
                      fid[j]+3.8*np.sqrt(covariance[1,1])]
        ax.axvline(fid[i],color="black",ls=":",lw=0.8)
        ax.axhline(fid[j],color="black",ls=":",lw=0.8)
        ax.set_xlim(max(prior_lo[i],min(all_x)),min(prior_hi[i],max(all_x)))
        ax.set_ylim(max(prior_lo[j],min(all_y)),min(prior_hi[j],max(all_y)))
        ax.set_xlabel(labels[i])
        ax.set_ylabel(labels[j])
        ax.grid(alpha=0.18)
    handles = [Line2D([0],[0],color=CASE_COLOR[c],lw=1.5,label=CASE_LABEL[c]) for c in CASES]
    axes[0,0].legend(handles=handles,frameon=False)
    fig.suptitle(f"{SCOPE_LABEL[scope]} covariance: noise-case comparison\n"
                 "solid 68%, dashed 95%; 9D prior-regularized covariance",y=1.015)
    fig.tight_layout()
    output = FIG/f"selected_constraints_noise_comparison_{scope}.jpg"
    fig.savefig(output,dpi=300)
    plt.show()
    return output

comparison_paths = [compare_noise(scope) for scope in SCOPES]

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(18/2.54,8.5/2.54),sharey=True)
x = np.arange(9)
sigma_rows = []
for ax,scope in zip(axes,SCOPES):
    for case,offset in zip(CASES,(-0.12,0.12)):
        result = results[(case,scope)]
        sigma = np.sqrt(np.diag(result["cov_q"]))
        ax.plot(x+offset,sigma,"o-",ms=4,lw=1,color=CASE_COLOR[case],label=CASE_LABEL[case])
        sigma_rows += [{"noise":CASE_LABEL[case],"covariance":SCOPE_LABEL[scope],
                        "parameter":name,"sigma_over_prior_width":value,
                        "estimable_fraction":result["estimable"][idx[name]]}
                       for name,value in zip(names,sigma)]
    ax.axhline(1/np.sqrt(12),color="black",ls=":",lw=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(labels,rotation=55,ha="right")
    ax.set_title(SCOPE_LABEL[scope])
    ax.set_xlabel("Parameter")
    ax.grid(axis="y",alpha=0.22)
axes[0].set_ylabel(r"$\sigma(\theta_i)/\Delta\theta_{i,{\rm prior}}$")
axes[0].legend(frameon=False)
fig.suptitle("Marginalized prior-regularized widths\n"
             "dotted line: standard deviation of the uniform prior",y=1.02)
fig.tight_layout()
sigma_path = FIG/"marginalized_sigma_over_prior_width.jpg"
fig.savefig(sigma_path,dpi=300)
plt.show()
sigma_table = pd.DataFrame(sigma_rows)
sigma_table.to_csv(FIG/"marginalized_sigma_over_prior_width.csv",index=False)
print(sigma_table.to_string(index=False))

In [ ]:
fig,axes = plt.subplots(2,2,figsize=(18/2.54,15.5/2.54))
mode_rows = []
for ax,((case,scope),result) in zip(axes.flat,results.items()):
    values,vectors = np.linalg.eigh(result["fisher_q"])
    order = np.argsort(values)[::-1]
    values,vectors = values[order],vectors[:,order]
    keep = np.arange(len(values)) < int(result["identified"].sum())
    image = ax.imshow(vectors.T,cmap="RdBu_r",vmin=-1,vmax=1,aspect="auto")
    ax.set_xticks(np.arange(9))
    ax.set_xticklabels(labels,rotation=55,ha="right")
    ax.set_yticks(np.arange(9))
    ax.set_yticklabels([f"{k+1}{'' if keep[k] else ' (null)'}" for k in range(9)])
    ax.set_title(f"{CASE_LABEL[case]}, {SCOPE_LABEL[scope]}")
    ax.set_xlabel("Normalized parameter")
    ax.set_ylabel("Fisher mode")
    for k,(value,vector,identified) in enumerate(zip(values,vectors.T,keep)):
        dominant = np.argsort(np.abs(vector))[::-1][:3]
        mode_rows.append({
            "noise":CASE_LABEL[case],"covariance":SCOPE_LABEL[scope],
            "mode":k+1,"identified":bool(identified),"eigenvalue":value,
            "dominant_parameters":", ".join(f"{names[q]} ({vector[q]:+.2f})" for q in dominant)})
fig.colorbar(image,ax=axes.ravel().tolist(),shrink=0.78,label="Eigenvector coefficient")
fig.suptitle("Fisher modes in prior-width-normalized parameter space")
fig.subplots_adjust(left=0.10,right=0.91,bottom=0.12,top=0.91,wspace=0.30,hspace=0.42)
mode_path = FIG/"fisher_eigenmodes.jpg"
fig.savefig(mode_path,dpi=300)
plt.show()
mode_table = pd.DataFrame(mode_rows)
mode_table.to_csv(FIG/"fisher_eigenmodes.csv",index=False)
print("Null modes:")
print(mode_table[~mode_table["identified"]].to_string(index=False))

## Reading the outputs

1. Start with the two p0_beta_constraints noise figures. They are the clearest marginalized \(P_0\)-\(\beta\) constraints.
2. Use the 12 separate triangle JPGs for within-case parameter groups. This avoids hiding contours beneath four overlaid cases.
3. Compare baseline and goal only within one covariance prescription using the selected_constraints_noise_comparison figures.
4. Check marginalized_sigma_over_prior_width.jpg. A width near \(1/\sqrt{12}\) remains close to the input uniform prior.
5. Check fisher_eigenmodes.jpg before quoting constraints. Three null modes mean the data alone cannot identify all nine simultaneous combinations.
6. Conditional-noise and Gaussian-total answer different forecasting questions. Neither includes connected non-Gaussian tSZ covariance or exact mask mode coupling.

## Battaglia12 profile versus the training distribution

This diagnostic compares the validated Battaglia12 baseline-deproj0 observation with the raw, linearly binned masked pseudo-\(D_\ell\) values used by the NPE. No logarithm, floor, asinh, or standardization is applied in this figure. The blue band and random curves come from the same greater-than-500k prepared dataset used for training.


In [ ]:
import os

PROFILE_RANDOM_COUNT = 8
PROFILE_REFERENCE_COUNT = 100_000
PROFILE_RANDOM_SEED = 20260901
PROFILE_DATASET_OVERRIDE = os.environ.get("SO_PREPARED_DATASET", "").strip()

profile_candidates = []
if PROFILE_DATASET_OVERRIDE:
    profile_candidates.append(Path(PROFILE_DATASET_OVERRIDE).expanduser())

provenance_path = ROOT / "input_provenance.json"
if provenance_path.exists():
    provenance = json.loads(provenance_path.read_text())
    provenance_dataset = Path(provenance["prepared_dataset"]).expanduser()
    profile_candidates.extend([
        provenance_dataset,
        Path(str(provenance_dataset).replace("/misc/home/", "/home/")),
    ])

dataset_name = (
    "so_masked_baseline_noise_cross_deproj0_ell80_7979_sbi_run.npz"
)
profile_candidates.extend([
    ROOT.parents[1] / "data_for_cluster"
    / "adrian_so_sbi_cases_ell80_7979_dataset_row_sobolrow"
    / dataset_name,
    Path("/home/kristero10/HalfDome_kSZ/SBI_analysis/data_for_cluster")
    / "adrian_so_sbi_cases_ell80_7979_dataset_row_sobolrow"
    / dataset_name,
    Path("/misc/home/kristero10/HalfDome_kSZ/SBI_analysis/data_for_cluster")
    / "adrian_so_sbi_cases_ell80_7979_dataset_row_sobolrow"
    / dataset_name,
])

deduplicated_candidates = []
for candidate in profile_candidates:
    candidate = candidate.resolve()
    if candidate not in deduplicated_candidates:
        deduplicated_candidates.append(candidate)

PROFILE_DATASET = next(
    (candidate for candidate in deduplicated_candidates if candidate.is_file()),
    None,
)
if PROFILE_DATASET is None:
    searched = "\n".join(f"  {candidate}" for candidate in deduplicated_candidates)
    raise FileNotFoundError(
        "Could not find the prepared >500k dataset. Set SO_PREPARED_DATASET "
        "to its cluster or local path. Searched:\n" + searched
    )

profile_contract_candidates = [
    ROOT.parent / "matched_baseline0_deproj_comparison"
    / "npe_conditioning_contract_N523788.npz",
    Path(
        "/lustre/work/kristero10/adrian_fisher_baseline_deproj0/"
        "battaglia12_baseline_deproj0_observation/prepared/"
        "battaglia12_masked_baseline_noise_cross_deproj0_sbi_observation.npz"
    ),
]
PROFILE_BATTAGLIA = next(
    (candidate.resolve() for candidate in profile_contract_candidates if candidate.is_file()),
    None,
)
if PROFILE_BATTAGLIA is None:
    raise FileNotFoundError(
        "Missing the validated Battaglia12 observation contract. Checked: "
        + ", ".join(str(path) for path in profile_contract_candidates)
    )

with np.load(PROFILE_DATASET, allow_pickle=True) as profile_data:
    required = {"x", "theta", "ell_binned", "param_names", "product"}
    missing = sorted(required.difference(profile_data.files))
    if missing:
        raise KeyError(f"Prepared dataset lacks keys: {missing}")
    training_profiles = np.asarray(profile_data["x"], dtype=np.float32)
    training_theta = np.asarray(profile_data["theta"], dtype=np.float32)
    profile_ell = np.asarray(profile_data["ell_binned"], dtype=float)
    profile_param_names = [str(value) for value in profile_data["param_names"]]
    profile_product = str(np.asarray(profile_data["product"]).reshape(()).item())

if training_profiles.shape[0] < 500_000:
    raise ValueError(
        f"Expected the >500k dataset, found {training_profiles.shape[0]:,} rows."
    )
if profile_product != "masked_baseline_noise_cross_deproj0":
    raise ValueError(f"Unexpected prepared product: {profile_product!r}")

with np.load(PROFILE_BATTAGLIA, allow_pickle=True) as battaglia_data:
    for key in ("binned_dell", "x_binned_dell", "obs", "x"):
        if key in battaglia_data.files:
            battaglia_profile = np.asarray(
                battaglia_data[key],
                dtype=np.float32,
            ).reshape(-1)
            break
    else:
        raise KeyError(f"No binned D_ell key in {PROFILE_BATTAGLIA}")

if battaglia_profile.shape != (training_profiles.shape[1],):
    raise ValueError(
        f"Battaglia shape {battaglia_profile.shape} does not match "
        f"training x shape {training_profiles.shape}."
    )
if profile_ell.shape != battaglia_profile.shape:
    raise ValueError(
        f"ell shape {profile_ell.shape} does not match D_ell "
        f"shape {battaglia_profile.shape}."
    )

profile_rng = np.random.default_rng(PROFILE_RANDOM_SEED)
reference_count = min(PROFILE_REFERENCE_COUNT, training_profiles.shape[0])
reference_indices = profile_rng.choice(
    training_profiles.shape[0],
    size=reference_count,
    replace=False,
)
line_indices = profile_rng.choice(
    training_profiles.shape[0],
    size=PROFILE_RANDOM_COUNT,
    replace=False,
)
reference_profiles = np.asarray(
    training_profiles[reference_indices],
    dtype=np.float32,
)
profile_quantiles = np.quantile(
    reference_profiles,
    [0.05, 0.50, 0.95],
    axis=0,
)
battaglia_bin_percentile = (
    100.0 * np.mean(reference_profiles <= battaglia_profile[None, :], axis=0)
)

fig, axes = plt.subplots(
    2,
    1,
    figsize=(18.0 / 2.54, 12.0 / 2.54),
    sharex=True,
    gridspec_kw={"height_ratios": [3.0, 1.25]},
)

axes[0].fill_between(
    profile_ell,
    profile_quantiles[0],
    profile_quantiles[2],
    color="#4C78A8",
    alpha=0.16,
    label="training 5--95%",
)
axes[0].plot(
    profile_ell,
    profile_quantiles[1],
    color="#4C78A8",
    lw=1.1,
    label="training median",
)
for plot_number, row_index in enumerate(line_indices):
    axes[0].plot(
        profile_ell,
        training_profiles[row_index],
        color="#4C78A8",
        lw=0.55,
        alpha=0.22,
        label="random training profiles" if plot_number == 0 else None,
    )
axes[0].plot(
    profile_ell,
    battaglia_profile,
    color="black",
    lw=1.5,
    zorder=10,
    label="Battaglia12",
)
axes[0].axhline(0.0, color="0.35", lw=0.6)
axes[0].set_ylabel(r"$D_\ell$")
axes[0].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
axes[0].grid(alpha=0.2)
axes[0].legend(frameon=False, ncol=2)

axes[1].plot(
    profile_ell,
    battaglia_bin_percentile,
    color="black",
    lw=1.1,
)
axes[1].axhspan(5.0, 95.0, color="#4C78A8", alpha=0.10)
axes[1].axhline(5.0, color="#4C78A8", lw=0.7, ls=":")
axes[1].axhline(95.0, color="#4C78A8", lw=0.7, ls=":")
axes[1].set_ylim(-2.0, 102.0)
axes[1].set_xlabel(r"$\ell$")
axes[1].set_ylabel("Battaglia12\npercentile [%]")
axes[1].grid(alpha=0.2)

fig.tight_layout()
profile_figure_path = FIG / "battaglia12_vs_training_profiles.jpg"
fig.savefig(profile_figure_path, dpi=350)
plt.show()

selected_profile_table = pd.DataFrame(
    training_theta[line_indices],
    columns=profile_param_names,
)
selected_profile_table.insert(0, "dataset_index", line_indices)
display(selected_profile_table)

print("Prepared dataset:", PROFILE_DATASET)
print("Prepared product:", profile_product)
print("Training rows:", f"{training_profiles.shape[0]:,}")
print("Battaglia12 contract:", PROFILE_BATTAGLIA)
print("Saved:", profile_figure_path)


## Nine-parameter Fisher versus Battaglia12 SBI

The Fisher and SBI curves below use the same validated noisy Battaglia12 baseline-deproj0 observation and the same hard bounded prior. The Fisher likelihood is the saved linearized Battaglia12 model with the baseline conditional-noise covariance. The saved MAF chain is retained as a diagnostic only because its raw flow has effectively zero support inside the prior at this observation.


In [ ]:
from getdist import MCSamples, plots
import hashlib
import json
import warnings

# The full 9-parameter MAF run and its exact asinh(x/s) transform.
SBI_RUN = ROOT.parents[1] / "convergence_tests" / "N523788"
SBI_METADATA = SBI_RUN / "run_metadata.json"
SBI_TRANSFORM = SBI_RUN / "x_transform.npz"

# Use the validated masked baseline-noise deproj0 Battaglia12 contract. The
# legacy analysis/fiducial_battaglia12_dell.npy is a clean Fisher fiducial and
# is not a valid conditioning vector for this noisy NPE.
MATCHED_DIR = ROOT.parent / "matched_baseline0_deproj_comparison"
BATTAGLIA_CONTRACT = MATCHED_DIR / "npe_conditioning_contract_N523788.npz"
BATTAGLIA_PREFLIGHT = MATCHED_DIR / "npe_sampling_preflight_N523788.json"
BATTAGLIA_MCMC_SAMPLES = MATCHED_DIR / "sbi_samples_N523788.npy"

for required in (
    SBI_METADATA,
    SBI_TRANSFORM,
    BATTAGLIA_CONTRACT,
    BATTAGLIA_PREFLIGHT,
    BATTAGLIA_MCMC_SAMPLES,
):
    if not required.exists():
        raise FileNotFoundError(f"Missing matched Battaglia12 NPE input: {required}")

metadata = json.loads(SBI_METADATA.read_text())
SBI_N_TRAIN = int(metadata["n_train"])
if SBI_N_TRAIN != 523_788:
    raise ValueError(f"Expected N=523,788, found N={SBI_N_TRAIN:,}.")
if metadata.get("density_estimator") != "maf":
    raise ValueError(f"Expected MAF, found {metadata.get('density_estimator')!r}.")
if metadata.get("x_rescale_mode") != "asinh":
    raise ValueError(
        f"Expected asinh(x/s), found {metadata.get('x_rescale_mode')!r}."
    )

with np.load(SBI_TRANSFORM, allow_pickle=True) as z:
    transform_mode = str(np.asarray(z["mode"]).reshape(()).item())
    transform_scale = np.asarray(z["scale"], dtype=np.float32)
if transform_mode != "asinh":
    raise ValueError(f"Expected saved asinh transform, found {transform_mode!r}.")

with np.load(BATTAGLIA_CONTRACT, allow_pickle=True) as z:
    battaglia_dell = np.asarray(z["binned_dell"], dtype=np.float32)
    SBI_OBS_TRANSFORMED = np.asarray(
        z["transformed_observation"],
        dtype=np.float32,
    )
    contract_run = str(np.asarray(z["npe_run_dir"]).reshape(()).item())
    contract_observation = str(
        np.asarray(z["battaglia_observation"]).reshape(()).item()
    )
    contract_validation = str(
        np.asarray(z["validation_report"]).reshape(()).item()
    )

if Path(contract_run).name != SBI_RUN.name:
    raise ValueError(
        f"Conditioning contract is for {contract_run}, not {SBI_RUN.name}."
    )
if battaglia_dell.shape != transform_scale.shape:
    raise ValueError(
        f"Battaglia12 D_ell shape {battaglia_dell.shape} does not match "
        f"the NPE scale shape {transform_scale.shape}."
    )
recomputed_x = np.arcsinh(battaglia_dell / transform_scale).astype(np.float32)
if not np.allclose(
    recomputed_x,
    SBI_OBS_TRANSFORMED,
    rtol=2e-6,
    atol=2e-6,
):
    raise ValueError(
        "The matched Battaglia12 observation does not reproduce the saved "
        "asinh(x/s) conditioning vector."
    )

preflight = json.loads(BATTAGLIA_PREFLIGHT.read_text())
if preflight.get("sampling_method") != "mcmc":
    raise ValueError(
        "Expected prior-restricted MCMC samples, found "
        f"{preflight.get('sampling_method')!r}."
    )
if int(preflight.get("n_train", -1)) != SBI_N_TRAIN:
    raise ValueError("MCMC preflight and NPE metadata use different N_train.")

# Direct rejection is intentionally not attempted. For this observation the
# preflight found no in-prior proposals from 10,000 raw flow draws; MCMC samples
# the same prior-restricted learned density without waiting for direct accepts.
raw_inside_fraction = float(
    preflight.get("leakage", {}).get("fraction_within_prior", np.nan)
)
if not np.isfinite(raw_inside_fraction) or raw_inside_fraction < 1e-4:
    warnings.warn(
        "The MAF direct-proposal acceptance for this Battaglia12 observation "
        f"is {raw_inside_fraction:.3%}. Using the saved prior-restricted MCMC "
        "chain. This severe leakage is also a scientific reliability warning: "
        "interpret the SBI contour as an extrapolative NPE result."
    )

sbi_samples = np.asarray(
    np.load(BATTAGLIA_MCMC_SAMPLES),
    dtype=float,
)
if sbi_samples.ndim != 2 or sbi_samples.shape[1] != len(names):
    raise ValueError(f"Invalid MCMC sample shape {sbi_samples.shape}.")
if not np.all(np.isfinite(sbi_samples)):
    raise ValueError("Battaglia12 SBI MCMC samples contain non-finite values.")

inside_prior = np.all(
    (sbi_samples >= prior_lo.reshape(1, -1))
    & (sbi_samples <= prior_hi.reshape(1, -1)),
    axis=1,
)
if not np.all(inside_prior):
    raise ValueError(
        f"Only {inside_prior.mean():.3%} of MCMC samples are inside the "
        "training prior."
    )

SBI_TRUTH = np.asarray(fid, dtype=float)
SBI_OBS_SOURCE = "validated masked baseline-noise deproj0 Battaglia12 profile"
SBI_SAME_FIDUCIAL_AS_FISHER = True
SBI_SAME_OBSERVATION_AS_FISHER = True
observation_hash = hashlib.sha256(
    np.ascontiguousarray(SBI_OBS_TRANSFORMED).view(np.uint8)
).hexdigest()

print("NPE run:", SBI_RUN)
print("density estimator:", metadata["density_estimator"])
print("N_train:", f"{SBI_N_TRAIN:,}")
print("observation:", SBI_OBS_SOURCE)
print("source observation:", contract_observation)
print("source validation:", contract_validation)
print("observation hash:", observation_hash)
print(
    "transformed observation range:",
    (float(SBI_OBS_TRANSFORMED.min()), float(SBI_OBS_TRANSFORMED.max())),
)
print("posterior sampler: MCMC", preflight.get("mcmc_method"))
print("posterior samples:", sbi_samples.shape)
print("fraction inside prior:", f"{inside_prior.mean():.3%}")
print("raw direct-proposal acceptance:", f"{raw_inside_fraction:.3%}")


In [ ]:
import sys

FISHER_CASE = ("baseline", "conditional_noise")
FISHER_COVARIANCE_SCOPE = "conditional_noise"
N_FISHER_PROPOSALS = 1_000_000
N_FISHER_SAMPLES = 200_000
MINIMUM_FISHER_IMPORTANCE_ESS = 5_000

MATCHED_FISHER_DIR = ROOT / "matched_conditional_fisher"
MATCHED_FISHER_STEM = (
    "matched_fisher_baseline_conditional_noise_hard_uniform"
)
MATCHED_FISHER_BUNDLE = MATCHED_FISHER_DIR / f"{MATCHED_FISHER_STEM}.npz"
MATCHED_FISHER_SUMMARY = (
    MATCHED_FISHER_DIR / f"{MATCHED_FISHER_STEM}_summary.json"
)

if not MATCHED_FISHER_BUNDLE.exists():
    analysis_module_dir = ROOT.parents[1]
    if str(analysis_module_dir) not in sys.path:
        sys.path.insert(0, str(analysis_module_dir))
    from build_matched_so_fisher_posterior import build_matched_posterior

    build_matched_posterior(
        fisher_root=ROOT.parent,
        sensitivity_dir=ROOT,
        observation_path=BATTAGLIA_CONTRACT,
        output_dir=MATCHED_FISHER_DIR,
        covariance_scope=FISHER_COVARIANCE_SCOPE,
        proposal_draws=N_FISHER_PROPOSALS,
        posterior_samples=N_FISHER_SAMPLES,
        minimum_importance_ess=MINIMUM_FISHER_IMPORTANCE_ESS,
        seed=271828,
    )

with np.load(MATCHED_FISHER_BUNDLE, allow_pickle=True) as matched_fisher:
    fisher_samples_9d = np.asarray(matched_fisher["samples"], dtype=float)
    fisher_names = [str(value) for value in matched_fisher["param_names"]]
    fisher_fiducial = np.asarray(matched_fisher["fiducial"], dtype=float)
    fisher_prior_low = np.asarray(matched_fisher["prior_low"], dtype=float)
    fisher_prior_high = np.asarray(matched_fisher["prior_high"], dtype=float)
    fisher_observation = np.asarray(matched_fisher["observation"], dtype=float)
    fisher_matrix_matched = np.asarray(
        matched_fisher["fisher_matrix"],
        dtype=float,
    )

if fisher_names != names:
    raise ValueError(f"Fisher parameter order {fisher_names} != {names}.")
if not np.allclose(fisher_fiducial, fid, rtol=0.0, atol=1e-10):
    raise ValueError("Matched Fisher and notebook use different fiducials.")
if not np.allclose(fisher_prior_low, prior_lo, rtol=0.0, atol=1e-10):
    raise ValueError("Matched Fisher and SBI use different prior lower bounds.")
if not np.allclose(fisher_prior_high, prior_hi, rtol=0.0, atol=1e-10):
    raise ValueError("Matched Fisher and SBI use different prior upper bounds.")
if not np.allclose(
    fisher_observation,
    battaglia_dell,
    rtol=2e-6,
    atol=2e-6,
):
    raise ValueError("Matched Fisher and SBI are conditioned on different D_ell.")
if fisher_samples_9d.ndim != 2 or fisher_samples_9d.shape[1] != len(names):
    raise ValueError(f"Invalid matched Fisher sample shape {fisher_samples_9d.shape}.")

fisher_cov = np.cov(fisher_samples_9d, rowvar=False)
fisher_rank = int(
    np.count_nonzero(
        np.linalg.eigvalsh(fisher_matrix_matched)
        > np.linalg.eigvalsh(fisher_matrix_matched).max() * 1e-8
    )
)

rows = []
for method, chain in [
    ("Fisher conditional noise + hard prior", fisher_samples_9d),
    (f"SBI MAF N={SBI_N_TRAIN:,} [diagnostic]", sbi_samples),
]:
    mean = chain.mean(axis=0)
    std = chain.std(axis=0, ddof=1)
    q16, q84 = np.quantile(chain, [0.16, 0.84], axis=0)
    for i, name in enumerate(names):
        rows.append({
            "method": method,
            "conditioning_point": "same noisy Battaglia12 observation",
            "parameter": name,
            "truth": fid[i],
            "mean": mean[i],
            "std": std[i],
            "q16": q16[i],
            "q84": q84[i],
            "mean_minus_truth_over_std": (mean[i] - fid[i]) / std[i],
            "std_over_prior_width": std[i] / prior_width[i],
        })

fisher_sbi_9d_summary = pd.DataFrame(rows)
summary_9d_path = (
    FIG
    / "fisher_conditional_matched_vs_sbi_N523788_battaglia12_9param_summary.csv"
)
fisher_sbi_9d_summary.to_csv(summary_9d_path, index=False)
display(fisher_sbi_9d_summary)

print("Fisher covariance:", FISHER_COVARIANCE_SCOPE)
print("Fisher data rank:", f"{fisher_rank}/{len(names)}")
print("Fisher posterior samples:", fisher_samples_9d.shape)
print("Matched Fisher summary:", MATCHED_FISHER_SUMMARY)


In [ ]:
gd_labels = [LABEL.get(name, name).strip("$") for name in names]
gd_ranges = {
    name: [float(lo), float(hi)]
    for name, lo, hi in zip(names, prior_lo, prior_hi)
}

fisher_label = (
    "Fisher: baseline conditional noise, matched observation + hard prior"
)
sbi_label = (
    rf"SBI MAF: $N_{{\rm train}}={SBI_N_TRAIN:,}$ "
    "[failed raw-support diagnostic]"
)

gd_fisher = MCSamples(
    samples=fisher_samples_9d,
    names=names,
    labels=gd_labels,
    label=fisher_label,
    ranges=gd_ranges,
)
gd_sbi = MCSamples(
    samples=sbi_samples,
    names=names,
    labels=gd_labels,
    label=sbi_label,
    ranges=gd_ranges,
)
for gd_sample in (gd_fisher, gd_sbi):
    gd_sample.updateSettings({
        "smooth_scale_1D": 0.3,
        "smooth_scale_2D": 0.3,
        "fine_bins": 2048,
        "fine_bins_2D": 1024,
    })

g = plots.get_subplot_plotter(width_inch=18.0 / 2.54)
g.settings.axes_fontsize = 5.5
g.settings.lab_fontsize = 7
g.settings.legend_fontsize = 6.5
g.settings.alpha_filled_add = 0.35
g.settings.linewidth = 1.0
g.settings.num_plot_contours = 2
g.settings.figure_legend_frame = False
g.settings.scaling = False

FISHER_COLOR = "#355C7D"
SBI_COLOR = "#C14953"

# A single GetDist call preserves both roots. Fisher is unfilled and assigned
# the larger z-order; the final SBI root is the only filled sample set.
g.triangle_plot(
    [gd_fisher, gd_sbi],
    params=names,
    filled=[False, True],
    legend_labels=[fisher_label, sbi_label],
    contour_colors=[FISHER_COLOR, SBI_COLOR],
    contour_args=[
        {
            "filled": False,
            "color": FISHER_COLOR,
            "lw": 1.15,
            "ls": "--",
            "zorder": 30,
        },
        {
            "filled": True,
            "color": SBI_COLOR,
            "alpha": 0.38,
            "lw": 1.25,
            "ls": "-",
            "zorder": 10,
        },
    ],
    line_args=[
        {
            "color": FISHER_COLOR,
            "lw": 1.15,
            "ls": "--",
            "zorder": 30,
        },
        {
            "color": SBI_COLOR,
            "lw": 1.25,
            "ls": "-",
            "zorder": 10,
        },
    ],
    markers=SBI_TRUTH,
    marker_args={"color": "black", "lw": 0.8, "ls": ":"},
)

figure = plt.gcf()
if raw_inside_fraction < 0.01:
    figure.suptitle(
        "Diagnostic only: saved SBI flow failed the raw in-prior support check",
        color="#8B1E1E",
        fontsize=9,
        y=0.997,
    )
    figure.subplots_adjust(top=0.94)

corner_9d_path = (
    FIG
    / "fisher_conditional_matched_vs_sbi_N523788_battaglia12_9param.jpg"
)
figure.savefig(corner_9d_path, dpi=350)
plt.show()

print("Saved:", corner_9d_path)
print("Saved:", summary_9d_path)


### Interpretation

Black dotted markers are the Battaglia12 truth. Blue dashed contours are the linear Fisher posterior evaluated with the **same noisy Battaglia12 data vector**, the baseline conditional-noise covariance, and the same hard bounded uniform prior as SBI. The Fisher likelihood is shifted by the observed noisy residual rather than forced to remain centered at the clean fiducial.

The filled red MAF contour is diagnostic only: its raw flow places effectively no samples inside the training prior for this observation, so the prior-restricted MCMC chain does not establish a reliable NPE posterior. A publishable SBI/Fisher comparison requires a retrained NPE to pass the context-distribution, raw-support, and posterior calibration checks first.
